# Stage-2 VLM throughput benchmark (T4)

> **Set the runtime to GPU first:** Runtime → Change runtime type → **T4 GPU** → Save.
> That restarts the runtime, so run the cells from the top afterwards.

Settles the three parameters blocking the architecture (docs/architecture.md 5):

1. **3B or 7B?**
2. **Frames per window** — 4 / 8 / 16
3. **Resolution cap** — the term that turns out to dominate

### Why resolution is swept

At Qwen2.5-VL's default settings one 1280x720 frame costs **1,196 vision tokens**
(patch 14, merge 2 → 784 px/token; smart_resize maps 720p to 1288x728). An 8-frame window
is therefore **9,568 tokens** before the prompt. Capping to 256 tok/frame gives 2,016.
That ~10x swing in sequence length is almost certainly the dominant latency term, so
benchmarking at the default alone would measure a config we would never deploy.

Frames are pre-resized to exact multiples of 28, so the intended token count is what the
processor actually produces — cell 6 asserts this rather than assuming it.

Self-contained: no dataset, no repo upload. Throughput depends on model size, token count
and frame count, not on image content.

**Paste back the tables from the last two cells.**

In [2]:
# 1. Confirm a GPU is actually attached.
# `nvidia-smi` is absent entirely on a CPU runtime, so treat that as the signal.
import shutil, subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi not found -> this runtime has no GPU attached.\n")

import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit(
        "NO GPU ATTACHED.\n"
        "  Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save.\n"
        "  That restarts the runtime, so re-run from cell 1 (including the install cell)."
    )

p = torch.cuda.get_device_properties(0)
print(f"device: {p.name}  |  {p.total_memory/1e9:.1f} GB  |  SM{p.major}{p.minor}")
print("bf16 supported:", torch.cuda.is_bf16_supported(), " (T4 is SM75 -> expect False, so fp16)")

nvidia-smi not found -> this runtime has no GPU attached.

torch: 2.11.0+cpu | cuda available: False


SystemExit: NO GPU ATTACHED.
  Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save.
  That restarts the runtime, so re-run from cell 1 (including the install cell).

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# 2. Install (~3-5 min cold)
%pip install -q unsloth
%pip install -q --no-deps --upgrade "transformers>=4.49" qwen-vl-utils accelerate
print("installed")

In [ ]:
# 3. Harness
import gc, json, math, statistics, time
import numpy as np
import torch
from PIL import Image

PATCH, MERGE = 14, 2
FACTOR = PATCH * MERGE                        # 28 -> image dims must be multiples of this
PX_PER_TOKEN = PATCH * PATCH * MERGE * MERGE  # 784 px per vision token
SRC_W, SRC_H = 1280, 720                      # dominant real resolution (121/144 clips)

ANOMALY_CLASSES = [
    "traffic_accident", "traffic_congestion", "stalled_or_broken_down_vehicle",
    "vehicle_blocking_traffic", "wrong_way_driving", "road_spill_or_debris",
    "waterlogging_or_flood", "fire", "smoke", "fighting_or_violence",
    "loitering_or_suspicious_presence",
]
SYSTEM_PROMPT = (
    "You are a real-time visual anomaly detector for city drone, CCTV and dashcam footage. "
    "Given a short sequence of frames from one time window, decide whether they show one of "
    "these anomalies: " + ", ".join(ANOMALY_CLASSES) + ", or normal if nothing of concern is "
    "happening. Most footage is ordinary and should be called normal. "
    'Reply with a single JSON object: {"is_anomaly": true|false, "class_name": "<label>"}.'
)
USER_PROMPT = "What is happening in this window?"


def dims_for_token_budget(tok_per_frame, src_w=SRC_W, src_h=SRC_H):
    """Largest 28-aligned box with the source aspect ratio fitting a token budget.
    None -> no cap, i.e. Qwen's own smart_resize of native 720p."""
    if tok_per_frame is None:
        return (round(src_w / FACTOR) * FACTOR, round(src_h / FACTOR) * FACTOR)
    beta = math.sqrt((src_w * src_h) / (tok_per_frame * PX_PER_TOKEN))
    w = max(FACTOR, math.floor(src_w / beta / FACTOR) * FACTOR)
    h = max(FACTOR, math.floor(src_h / beta / FACTOR) * FACTOR)
    return w, h


def expected_tokens(w, h):
    return (w * h) // PX_PER_TOKEN


_rng = np.random.default_rng(0)

def fake_frames(n, w, h):
    """Structured noise pre-resized to the target box. Flat colour can compress through the
    vision path and understate real cost."""
    return [Image.fromarray(_rng.integers(0, 255, (h, w, 3), dtype=np.uint8)) for _ in range(n)]


def free():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()


def load(model_id):
    from unsloth import FastVisionModel
    free()
    model, processor = FastVisionModel.from_pretrained(
        model_id, load_in_4bit=True, use_gradient_checkpointing=False
    )
    FastVisionModel.for_inference(model)
    return model, processor, torch.cuda.max_memory_allocated() / 1e9


def bench_config(model, processor, n_frames, tok_per_frame,
                 n_reps=5, n_warmup=2, max_new_tokens=24):
    """Time one (frames, resolution) config on an already-loaded model."""
    from qwen_vl_utils import process_vision_info
    free()
    w, h = dims_for_token_budget(tok_per_frame)
    frames = fake_frames(n_frames, w, h)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [{"type": "image", "image": f} for f in frames]
                                     + [{"type": "text", "text": USER_PROMPT}]},
    ]
    try:
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                           padding=True, return_tensors="pt").to(model.device)
        n_tok = int(inputs["input_ids"].shape[1])

        for _ in range(n_warmup):
            with torch.inference_mode():
                model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        torch.cuda.synchronize()

        times = []
        for _ in range(n_reps):
            t0 = time.perf_counter()
            with torch.inference_mode():
                model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
            torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)

        return {
            "n_frames": n_frames,
            "tok_per_frame": tok_per_frame if tok_per_frame else "default",
            "frame_wh": f"{w}x{h}",
            "expected_vision_tok": expected_tokens(w, h) * n_frames,
            "actual_input_tok": n_tok,
            "median_sec": statistics.median(times),
            "min_sec": min(times),
            "max_sec": max(times),
            "peak_gb": torch.cuda.max_memory_allocated() / 1e9,
        }
    except torch.cuda.OutOfMemoryError:
        print(f"    OOM: {n_frames}f @ {tok_per_frame} tok/frame"); free(); return None
    except Exception as e:
        print(f"    fail: {type(e).__name__}: {str(e)[:160]}"); free(); return None


# Sanity-check the token math before spending GPU time on it
for t in [None, 512, 256, 128]:
    w, h = dims_for_token_budget(t)
    print(f"cap={str(t):>7} -> {w}x{h} = {expected_tokens(w,h)} tok/frame")

In [ ]:
# 4. Sweep the 3B - primary candidate for a T4 runtime.
#    Loaded ONCE; every config runs against the same instance.
results = []

MODEL_3B = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"
print(f"loading {MODEL_3B} ...", flush=True)
model, processor, weights_gb = load(MODEL_3B)
print(f"  weights: {weights_gb:.2f} GB\n", flush=True)

# None = Qwen default (uncapped), included once at 8 frames to quantify what capping buys
CONFIGS = [(4, 128), (8, 128), (16, 128),
           (4, 256), (8, 256), (16, 256),
           (8, 512), (8, None)]

for n_frames, tok in CONFIGS:
    print(f"  3B: {n_frames}f @ {tok} tok/frame", flush=True)
    r = bench_config(model, processor, n_frames, tok)
    if r:
        r["model"] = "Qwen2.5-VL-3B"; r["weights_gb"] = weights_gb
        results.append(r)
        print(f"    {r['median_sec']:.3f}s  |  {r['actual_input_tok']} tok  |  peak {r['peak_gb']:.1f}GB",
              flush=True)

del model, processor
free()
print("\n3B done")

In [ ]:
# 5. 7B at the more promising configs only (larger download; may not fit)
MODEL_7B = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
try:
    print(f"loading {MODEL_7B} ...", flush=True)
    model, processor, weights_gb = load(MODEL_7B)
    print(f"  weights: {weights_gb:.2f} GB\n", flush=True)
    for n_frames, tok in [(4, 128), (8, 128), (8, 256), (16, 256)]:
        print(f"  7B: {n_frames}f @ {tok} tok/frame", flush=True)
        r = bench_config(model, processor, n_frames, tok)
        if r:
            r["model"] = "Qwen2.5-VL-7B"; r["weights_gb"] = weights_gb
            results.append(r)
            print(f"    {r['median_sec']:.3f}s  |  {r['actual_input_tok']} tok  |  peak {r['peak_gb']:.1f}GB",
                  flush=True)
    del model, processor
    free()
except Exception as e:
    print(f"7B unavailable on this GPU: {type(e).__name__}: {str(e)[:200]}")

with open("bench_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\ncollected {len(results)} configs")

In [ ]:
# 6. RESULTS
hdr = (f"{'model':<16}{'frames':>7}{'tok/frm':>9}{'frame wh':>11}{'input tok':>11}"
       f"{'sec/win':>9}{'peak GB':>9}")
print(hdr); print("-" * len(hdr))
for r in sorted(results, key=lambda x: (x["model"], x["median_sec"])):
    print(f"{r['model']:<16}{r['n_frames']:>7}{str(r['tok_per_frame']):>9}{r['frame_wh']:>11}"
          f"{r['actual_input_tok']:>11}{r['median_sec']:>9.3f}{r['peak_gb']:>9.1f}")

# Verify the resolution cap did what the token math predicted
print("\ntoken-math check (actual should be expected + ~160 prompt tokens):")
for r in results:
    delta = r["actual_input_tok"] - r["expected_vision_tok"]
    flag = "ok" if 0 < delta < 400 else "<-- CHECK"
    print(f"  {r['model']} {r['n_frames']}f@{r['tok_per_frame']}: "
          f"expected_vision={r['expected_vision_tok']} actual={r['actual_input_tok']} "
          f"delta={delta} {flag}")

In [ ]:
# 7. VERDICT - what the numbers mean for the pipeline
#
# Public test = 3391s of video. At 4s windows / 2s stride -> 1696 windows.
# Real-time for ONE feed: must process a window every `stride` seconds.
TEST_SECONDS, STRIDE = 3391, 2.0
TEST_WINDOWS = TEST_SECONDS / STRIDE

print(f"public test: {TEST_SECONDS}s -> {TEST_WINDOWS:.0f} windows @ {STRIDE}s stride")
print(f"real-time budget: {STRIDE:.1f}s per window per feed\n")

hdr = f"{'config':<34}{'offline test':>14}{'realtime':>10}{'gate drop needed':>18}"
print(hdr); print("-" * len(hdr))
for r in sorted(results, key=lambda x: x["median_sec"]):
    total_min = TEST_WINDOWS * r["median_sec"] / 60
    rt = r["median_sec"] <= STRIDE
    drop = max(0.0, 1 - STRIDE / r["median_sec"])   # fraction Stage 1 must discard
    name = f"{r['model']} {r['n_frames']}f@{r['tok_per_frame']}"
    print(f"{name:<34}{total_min:>11.1f} min{('YES' if rt else 'no'):>10}{drop*100:>17.0f}%")

print("\nReading this table:")
print("- 'offline test' = whole public test set, ungated. The leaderboard runs offline, so a")
print("   large number here is tolerable and NOT a blocker.")
print("- 'gate drop needed' = what Stage 1 must discard for one feed to hit real-time.")
print("   Above ~90% is unrealistic for a high-recall gate; prefer fewer frames or a")
print("   tighter resolution cap before giving up model size.")